In [2]:
!pip install -q -U langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.5/41.5 kB 1.6 MB/s eta 0:00:00


In [3]:
from google.colab import userdata
GOOGLE_API_KEY = userdata.get('Google_Api_Key_1')

In [4]:
from langchain_google_genai import ChatGoogleGenerativeAI

In [5]:
llm = ChatGoogleGenerativeAI(
    model = "gemini-2.0-flash-exp",
    api_key = GOOGLE_API_KEY
)

In [6]:
llm.invoke("Hello")

AIMessage(content='Hi there! How can I help you today?', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': [{'category': 'HARM_CATEGORY_HATE_SPEECH', 'probability': 'NEGLIGIBLE', 'blocked': False}, {'category': 'HARM_CATEGORY_DANGEROUS_CONTENT', 'probability': 'NEGLIGIBLE', 'blocked': False}, {'category': 'HARM_CATEGORY_HARASSMENT', 'probability': 'NEGLIGIBLE', 'blocked': False}, {'category': 'HARM_CATEGORY_SEXUALLY_EXPLICIT', 'probability': 'NEGLIGIBLE', 'blocked': False}]}, id='run-bc81abea-b7aa-4660-abe7-efb7f89bda05-0', usage_metadata={'input_tokens': 2, 'output_tokens': 11, 'total_tokens': 13, 'input_token_details': {'cache_read': 0}})

In [7]:
from langchain_core.tools import tool
import math

@tool
def annual_return(a: int) -> float:
    """Returns annual return of a number."""
    return (a + 90)/5 + 3

@tool
def add(a: int, b: int) -> int:
    """Return addition of two number a and b."""
    return a + b

@tool
def multiply(a: int, b: int) -> int:
    """Returns Multiplies of two number a and b."""
    return a * b

tools = [annual_return, add , multiply]

In [8]:
llm.invoke("What is the annual return of 100?")

AIMessage(content='The phrase "annual return of 100" is ambiguous. It could mean a few different things, depending on the context. Here\'s a breakdown of the possible interpretations:\n\n**1. 100% Annual Return:**\n\n* **Meaning:** This is the most likely interpretation when talking about investments. It means that for every dollar invested, you earn one dollar in profit over the course of a year.\n* **Calculation:** If you invested $100, a 100% annual return would mean you earn $100 in profit, bringing your total to $200.\n* **Example:** If you invest $100 and get a 100% return, at the end of the year you\'d have $200.\n* **Realism:** A consistent 100% annual return is **extremely rare and highly unlikely** in most legitimate investment scenarios. It\'s often a sign of very high risk or even a scam.\n\n**2. $100 Annual Return (Fixed Amount):**\n\n* **Meaning:** This means you earn a fixed amount of $100 every year, regardless of the initial investment. This is less common in investmen

In [11]:
llm_with_tools = llm.bind_tools(tools)

In [13]:
llm_with_tools.invoke("Add 5 and 2 then multiply it with 5?")

AIMessage(content='', additional_kwargs={'function_call': {'name': 'add', 'arguments': '{"a": 5.0, "b": 2.0}'}}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': [{'category': 'HARM_CATEGORY_HATE_SPEECH', 'probability': 'NEGLIGIBLE', 'blocked': False}, {'category': 'HARM_CATEGORY_DANGEROUS_CONTENT', 'probability': 'NEGLIGIBLE', 'blocked': False}, {'category': 'HARM_CATEGORY_HARASSMENT', 'probability': 'NEGLIGIBLE', 'blocked': False}, {'category': 'HARM_CATEGORY_SEXUALLY_EXPLICIT', 'probability': 'NEGLIGIBLE', 'blocked': False}]}, id='run-33ea8d26-e9f0-4418-a822-f6435a4bd3fc-0', tool_calls=[{'name': 'add', 'args': {'a': 5.0, 'b': 2.0}, 'id': 'ee9c01d5-e27f-41bc-b951-900334a7e4ad', 'type': 'tool_call'}], usage_metadata={'input_tokens': 154, 'output_tokens': 3, 'total_tokens': 157, 'input_token_details': {'cache_read': 0}})

In [20]:
from langchain_core.messages import HumanMessage, AIMessage
query = "What is addition of 90 and 5"
messages = [HumanMessage(query)]
display(messages)

[HumanMessage(content='What is addition of 90 and 5', additional_kwargs={}, response_metadata={})]

In [21]:
ai_msg = llm_with_tools.invoke(messages)
messages.append(ai_msg)
messages

[HumanMessage(content='What is addition of 90 and 5', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'function_call': {'name': 'add', 'arguments': '{"a": 90.0, "b": 5.0}'}}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': [{'category': 'HARM_CATEGORY_HATE_SPEECH', 'probability': 'NEGLIGIBLE', 'blocked': False}, {'category': 'HARM_CATEGORY_DANGEROUS_CONTENT', 'probability': 'NEGLIGIBLE', 'blocked': False}, {'category': 'HARM_CATEGORY_HARASSMENT', 'probability': 'NEGLIGIBLE', 'blocked': False}, {'category': 'HARM_CATEGORY_SEXUALLY_EXPLICIT', 'probability': 'NEGLIGIBLE', 'blocked': False}]}, id='run-468421ff-246d-4c7b-ba74-2b3719f2c0ac-0', tool_calls=[{'name': 'add', 'args': {'a': 90.0, 'b': 5.0}, 'id': '875d1c86-4315-4b08-bfa2-3ceb5779d495', 'type': 'tool_call'}], usage_metadata={'input_tokens': 151, 'output_tokens': 3, 'total_tokens': 154, 'input_token_details': {'cache_re

In [22]:
ai_msg.tool_calls

[{'name': 'add',
  'args': {'a': 90.0, 'b': 5.0},
  'id': '875d1c86-4315-4b08-bfa2-3ceb5779d495',
  'type': 'tool_call'}]

In [23]:
for tool_calls in ai_msg.tool_calls:
    selected_tool = {
        "add" : add,
        "annual_return" : annual_return,
        "multiply" : multiply
    }[tool_calls["name"].lower()]
    tool_msg = selected_tool.invoke(tool_calls)
    display(tool_msg)
    messages.append(tool_msg)

ToolMessage(content='95', name='add', tool_call_id='875d1c86-4315-4b08-bfa2-3ceb5779d495')

In [24]:
messages

[HumanMessage(content='What is addition of 90 and 5', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'function_call': {'name': 'add', 'arguments': '{"a": 90.0, "b": 5.0}'}}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': [{'category': 'HARM_CATEGORY_HATE_SPEECH', 'probability': 'NEGLIGIBLE', 'blocked': False}, {'category': 'HARM_CATEGORY_DANGEROUS_CONTENT', 'probability': 'NEGLIGIBLE', 'blocked': False}, {'category': 'HARM_CATEGORY_HARASSMENT', 'probability': 'NEGLIGIBLE', 'blocked': False}, {'category': 'HARM_CATEGORY_SEXUALLY_EXPLICIT', 'probability': 'NEGLIGIBLE', 'blocked': False}]}, id='run-468421ff-246d-4c7b-ba74-2b3719f2c0ac-0', tool_calls=[{'name': 'add', 'args': {'a': 90.0, 'b': 5.0}, 'id': '875d1c86-4315-4b08-bfa2-3ceb5779d495', 'type': 'tool_call'}], usage_metadata={'input_tokens': 151, 'output_tokens': 3, 'total_tokens': 154, 'input_token_details': {'cache_re

In [25]:
llm_with_tools.invoke(messages)

AIMessage(content='The addition of 90 and 5 is 95.', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': [{'category': 'HARM_CATEGORY_HATE_SPEECH', 'probability': 'NEGLIGIBLE', 'blocked': False}, {'category': 'HARM_CATEGORY_DANGEROUS_CONTENT', 'probability': 'NEGLIGIBLE', 'blocked': False}, {'category': 'HARM_CATEGORY_HARASSMENT', 'probability': 'NEGLIGIBLE', 'blocked': False}, {'category': 'HARM_CATEGORY_SEXUALLY_EXPLICIT', 'probability': 'NEGLIGIBLE', 'blocked': False}]}, id='run-d009ecbf-0a94-4d6e-adee-38a55046fe28-0', usage_metadata={'input_tokens': 186, 'output_tokens': 15, 'total_tokens': 201, 'input_token_details': {'cache_read': 0}})